# Sesión 4 · AC_PR_4_1 · Intercomparación de los loggers
**Sistemas de adquisición de datos · TSU Electrónica y Pruebas · Jabil · 2027**

No necesitas programar: ejecuta cada celda con ▶ en orden y cambia solo los campos del formulario.

In [ ]:
# @title Preparación (ejecuta esta celda primero; no hay nada que cambiar)
import io, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (11, 4.2), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})
EXACTITUD = {"Govee": 0.3, "RC5": 0.5}   # °C, del fabricante
DT_S = 60
SIMULADOS = {"Eq01": [24.1, 24.1, 24.0, 24.2, 24.0, 24.1, 24.1, 24.1, 24.1, 24.2, 24.2, 24.3, 24.2, 24.2, 24.2, 24.2, 24.3, 24.2, 24.2, 24.2, 24.2, 24.2, 24.2, 24.3, 24.2, 24.2, 24.3, 24.2, 24.3, 24.2, 24.3, 24.3, 24.3, 24.3, 24.3, 24.2, 24.3, 24.4, 24.4, 24.4, 24.4, 24.4, 24.3, 24.3, 24.4, 24.4, 24.4, 24.4, 24.5, 24.4, 24.5, 24.5, 24.5, 24.4, 24.5, 24.5, 24.5, 24.4, 24.5, 24.4], "Eq02": [23.9, 23.8, 24.0, 23.9, 23.9, 24.0, 24.0, 23.9, 24.0, 23.9, 23.9, 24.0, 23.9, 23.9, 24.0, 24.0, 24.0, 24.0, 24.0, 24.0, 24.0, 24.0, 23.9, 24.1, 24.0, 24.0, 24.1, 24.0, 24.1, 24.2, 24.1, 24.1, 24.0, 24.1, 24.1, 24.2, 24.1, 24.1, 24.1, 24.2, 24.2, 24.2, 24.2, 24.1, 24.2, 24.1, 24.2, 24.2, 24.2, 24.2, 24.2, 24.3, 24.2, 24.2, 24.3, 24.2, 24.2, 24.3, 24.2, 24.3], "Eq03": [23.8, 23.9, 23.8, 23.9, 23.9, 24.0, 24.0, 23.9, 24.0, 23.9, 24.0, 23.9, 23.9, 24.0, 24.0, 23.9, 23.9, 24.0, 24.0, 23.9, 23.9, 24.0, 23.9, 24.1, 23.9, 24.0, 24.0, 24.1, 24.0, 24.1, 24.1, 24.1, 24.1, 24.1, 24.1, 24.0, 24.1, 24.0, 24.1, 24.1, 24.2, 24.1, 24.2, 24.1, 24.1, 24.1, 24.1, 24.2, 24.2, 24.2, 24.2, 24.3, 24.2, 24.3, 24.2, 24.2, 24.2, 24.3, 24.3, 24.2], "Eq04": [23.9, 23.9, 23.9, 23.9, 24.0, 24.0, 24.0, 24.0, 24.0, 23.9, 24.0, 23.9, 24.0, 24.0, 24.0, 24.0, 24.0, 24.1, 24.1, 24.0, 24.1, 24.1, 24.0, 24.1, 24.1, 24.1, 24.1, 24.0, 24.2, 24.1, 24.1, 24.2, 24.2, 24.2, 24.2, 24.2, 24.1, 24.2, 24.2, 24.1, 24.1, 24.2, 24.1, 24.3, 24.2, 24.2, 24.2, 24.2, 24.2, 24.2, 24.2, 24.3, 24.2, 24.3, 24.4, 24.4, 24.2, 24.2, 24.3, 24.3], "Eq05": [24.1, 24.2, 24.2, 24.2, 24.2, 24.2, 24.2, 24.3, 24.1, 24.2, 24.3, 24.2, 24.2, 24.2, 24.2, 24.3, 24.4, 24.3, 24.3, 24.2, 24.3, 24.3, 24.2, 24.3, 24.3, 24.4, 24.4, 24.4, 24.3, 24.3, 24.3, 24.4, 24.4, 24.4, 24.4, 24.4, 24.5, 24.5, 24.5, 24.3, 24.4, 24.4, 24.4, 24.5, 24.4, 24.5, 24.5, 24.5, 24.5, 24.5, 24.4, 24.5, 24.6, 24.6, 24.5, 24.6, 24.5, 24.5, 24.6, 24.5], "Eq06": [24.2, 24.1, 24.2, 24.3, 24.2, 24.3, 24.2, 24.1, 24.2, 24.3, 24.3, 24.2, 24.2, 24.3, 24.2, 24.3, 24.2, 24.3, 24.3, 24.3, 24.3, 24.3, 24.4, 24.2, 24.3, 24.3, 24.3, 24.4, 24.3, 24.3, 24.3, 24.3, 24.4, 24.4, 24.4, 24.3, 24.4, 24.4, 24.4, 24.4, 24.4, 24.4, 24.4, 24.5, 24.4, 24.5, 24.5, 24.5, 24.5, 24.4, 24.5, 24.5, 24.5, 24.5, 24.5, 24.6, 24.6, 24.6, 24.5, 24.5], "Eq07": [24.7, 24.7, 24.8, 24.8, 24.8, 24.8, 24.7, 24.9, 24.7, 24.8, 24.8, 24.9, 24.9, 24.9, 24.9, 24.9, 24.9, 24.8, 24.9, 24.8, 24.9, 24.9, 24.8, 24.9, 24.9, 24.9, 24.9, 24.9, 24.9, 25.0, 24.9, 25.0, 25.0, 25.1, 24.9, 24.9, 25.0, 24.9, 25.0, 25.1, 25.1, 25.0, 25.0, 25.0, 25.1, 25.0, 25.0, 25.1, 25.1, 25.1, 25.0, 25.2, 25.0, 25.2, 25.1, 25.1, 25.1, 25.1, 25.2, 25.1], "Eq08": [24.0, 24.0, 24.1, 24.1, 23.9, 24.1, 24.0, 24.1, 24.1, 24.0, 24.0, 24.1, 24.1, 24.0, 24.2, 24.1, 24.1, 24.2, 24.2, 24.1, 24.1, 24.2, 24.2, 24.1, 24.1, 24.1, 24.2, 24.2, 24.2, 24.2, 24.2, 24.2, 24.3, 24.2, 24.3, 24.2, 24.3, 24.2, 24.2, 24.2, 24.3, 24.3, 24.3, 24.3, 24.3, 24.3, 24.3, 24.3, 24.3, 24.3, 24.3, 24.3, 24.3, 24.4, 24.4, 24.3, 24.3, 24.4, 24.4, 24.4], "Eq09": [24.1, 24.1, 24.1, 24.1, 24.1, 24.2, 24.1, 24.2, 24.1, 24.1, 24.2, 24.2, 24.1, 24.1, 24.2, 24.2, 24.2, 24.1, 24.2, 24.1, 24.2, 24.2, 24.2, 24.2, 24.2, 24.3, 24.2, 24.3, 24.3, 24.3, 24.2, 24.3, 24.3, 24.2, 24.3, 24.3, 24.2, 24.3, 24.3, 24.2, 24.4, 24.4, 24.3, 24.3, 24.4, 24.3, 24.4, 24.3, 24.3, 24.4, 24.3, 24.3, 24.4, 24.4, 24.3, 24.4, 24.5, 24.5, 24.5, 24.4], "Eq10": [23.8, 24.0, 23.9, 23.9, 24.0, 23.9, 24.0, 24.0, 23.9, 24.0, 24.0, 23.9, 24.0, 23.9, 24.0, 24.1, 24.1, 24.1, 24.1, 24.0, 24.1, 24.1, 23.9, 24.1, 24.0, 24.1, 24.1, 24.1, 24.1, 24.1, 24.1, 24.0, 24.2, 24.2, 24.1, 24.2, 24.2, 24.2, 24.2, 24.1, 24.2, 24.2, 24.1, 24.2, 24.3, 24.3, 24.2, 24.2, 24.3, 24.2, 24.2, 24.2, 24.3, 24.3, 24.2, 24.3, 24.2, 24.3, 24.2, 24.3], "RC5_1": [23.9, 24.0, 23.9, 24.0, 24.1, 24.0, 24.0, 24.1, 24.1, 24.0, 24.0, 24.1, 23.9, 24.1, 24.1, 24.0, 24.0, 24.1, 24.1, 24.2, 24.1, 24.2, 24.1, 24.1, 24.1, 24.1, 24.2, 24.1, 24.1, 24.2, 24.3, 24.2, 24.2, 24.1, 24.0, 24.2, 24.2, 24.3, 24.2, 24.3, 24.2, 24.3, 24.3, 24.4, 24.2, 24.3, 24.3, 24.3, 24.4, 24.3, 24.2, 24.3, 24.4, 24.4, 24.3, 24.3, 24.4, 24.3, 24.4, 24.3], "RC5_2": [24.3, 24.3, 24.4, 24.2, 24.2, 24.4, 24.3, 24.3, 24.4, 24.3, 24.5, 24.3, 24.4, 24.4, 24.4, 24.3, 24.5, 24.3, 24.4, 24.4, 24.5, 24.4, 24.5, 24.4, 24.5, 24.4, 24.6, 24.5, 24.5, 24.6, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.6, 24.6, 24.6, 24.6, 24.6, 24.6, 24.5, 24.5, 24.6, 24.6, 24.6, 24.6, 24.6, 24.6, 24.7, 24.6, 24.7, 24.6, 24.6, 24.7], "RC5_3": [23.9, 23.8, 23.8, 24.0, 23.9, 23.9, 23.9, 24.0, 23.9, 23.9, 23.8, 23.9, 23.9, 23.9, 23.9, 23.9, 23.9, 23.9, 24.0, 24.0, 24.0, 24.0, 24.0, 24.0, 24.0, 24.1, 24.0, 24.0, 24.0, 24.1, 24.0, 24.1, 24.0, 24.1, 24.1, 24.1, 24.0, 24.0, 24.1, 24.2, 24.1, 24.1, 24.2, 24.2, 24.2, 24.0, 24.1, 24.1, 24.1, 24.1, 24.2, 24.2, 24.3, 24.2, 24.2, 24.2, 24.2, 24.3, 24.2, 24.2]}

def modelo(nombre):
    return "RC5" if re.search(r"rc[-_ ]?5", nombre, re.I) else "Govee"

def lee_csv(nombre, contenido):
    """Lee el CSV de un logger aunque cambien los nombres de columna o el separador."""
    txt = contenido.decode("utf-8-sig", errors="ignore")
    df = pd.read_csv(io.StringIO(txt), sep=None, engine="python")
    col_t = col_T = None
    for c in df.select_dtypes("object").columns:           # fecha y hora: solo columnas de texto
        if col_t is None and pd.to_datetime(df[c], errors="coerce", dayfirst=True).notna().mean() > 0.8:
            col_t = c
    for c in df.columns:                                   # columna de temperatura
        if re.search(r"temp|°c|tmp", str(c), re.I):
            col_T = c; break
    if col_T is None:
        col_T = [c for c in df.columns if c != col_t][0]
    T = pd.to_numeric(df[col_T].astype(str).str.replace(",", "."), errors="coerce")
    t = pd.to_datetime(df[col_t], errors="coerce", dayfirst=True)
    s = pd.Series(T.values, index=t).dropna().sort_index()
    if re.search(r"°F|\bF\b|fahr", str(col_T), re.I):     # por si la app exporta en °F
        s = (s - 32) * 5 / 9
    return s

print("Listo. Sigue con el bloque A.")


## Bloque A · Los registros
En clase elige **subir mis archivos** y selecciona los CSV de los 13 loggers (el profesor los comparte). Si todavía no hay registros, deja **simulados**: la práctica funciona igual y la gráfica lo indica.

In [ ]:
# @title Bloque A · Cargar los registros de la intercomparación
fuente = "simulados"  # @param ["simulados", "subir mis archivos"]
series = {}
if fuente == "subir mis archivos":
    from google.colab import files
    print("Selecciona los CSV de todos los loggers (nombres como Eq03_20270219_T.csv o RC5_1.csv)")
    for nombre, contenido in files.upload().items():
        series[nombre.split(".")[0]] = lee_csv(nombre, contenido)
    # ventana común: desde el último arranque hasta el primer final
    ini = max(s.index.min() for s in series.values()); fin = min(s.index.max() for s in series.values())
    if ini >= fin:
        raise SystemExit("Los registros no se traslapan en el tiempo: revisa que sean de la misma hora y fecha.")
    series = {k: s[(s.index >= ini) & (s.index <= fin)] for k, s in series.items()}
    print(f"Ventana común: {ini} a {fin}")
    etiqueta = "datos medidos"
else:
    series = {k: pd.Series(v) for k, v in SIMULADOS.items()}
    etiqueta = "DATOS SIMULADOS"
print(f"{len(series)} loggers cargados ({etiqueta}).")

fig, ax = plt.subplots()
for k, s in series.items():
    ax.plot(np.arange(len(s)) * DT_S / 60, s.values, lw=1.6 if modelo(k) == "RC5" else 1.1, label=k)
ax.set_xlabel("Tiempo [min]"); ax.set_ylabel("Temperatura [°C]"); ax.set_title(f"Intercomparación · {etiqueta}")
ax.legend(ncol=7, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False)
plt.show()


## Bloque B · ¿Quién cabe en su especificación?
Referencia de trabajo: la **mediana** de las medias. Un logger es compatible si |media − mediana| ≤ exactitud de su fabricante.

In [ ]:
# @title Bloque B · Tabla de la intercomparación y veredicto por logger
filas = []
for k, s in series.items():
    n = int(s.count()); m = float(s.mean()); sd = float(s.std(ddof=1))
    filas.append(dict(Logger=k, Modelo=modelo(k), n=n, Media=m, s=sd, u=sd / np.sqrt(n),
                      Exactitud=EXACTITUD[modelo(k)]))
tabla = pd.DataFrame(filas).sort_values("Logger").reset_index(drop=True)
mediana = float(tabla["Media"].median())
tabla["Desviación"] = tabla["Media"] - mediana
# s respecto al grupo: dispersión de la diferencia instante a instante contra la mediana
# (quita la deriva del salón, que es común a todos)
largo = min(len(s) for s in series.values())
M = np.column_stack([series[k].values[:largo] for k in tabla["Logger"]])
med_inst = np.median(M, axis=1)
tabla["s respecto al grupo"] = [float(np.std(M[:, i] - med_inst, ddof=1)) for i in range(M.shape[1])]
tabla["Veredicto"] = np.where(tabla["Desviación"].abs() <= tabla["Exactitud"], "Compatible", "Fuera respecto al grupo")
print(f"Mediana de las medias (referencia de trabajo): {mediana:.3f} °C")
print("Nota: s incluye la deriva del salón durante la hora; «s respecto al grupo» la quita.\n")
display(tabla.round({"Media": 3, "s": 4, "u": 4, "Desviación": 3, "s respecto al grupo": 4}))

fig, ax = plt.subplots()
x = np.arange(len(tabla))
ax.bar(x, 2 * tabla["Exactitud"], bottom=-tabla["Exactitud"], color="#E2F0D9", width=0.7, label="± exactitud")
col = np.where(tabla["Veredicto"] == "Compatible", "#2E7D32", "#C62828")
ax.scatter(x, tabla["Desviación"], c=col, s=90, zorder=3)
ax.axhline(0, color="#444", lw=1)
ax.set_xticks(x); ax.set_xticklabels(tabla["Logger"], rotation=45, ha="right")
ax.set_ylabel("Desviación de la mediana [°C]"); ax.set_title(etiqueta)
plt.show()


## Bloque C · Tu logger, contra tu cálculo
Pega la columna de temperatura de tu CSV en la pestaña «Práctica» de tu hoja (columna B), escribe la mediana del grupo y calcula. Después pega aquí los seis valores en la columna E: la columna F debe marcar ✔.

In [ ]:
# @title Bloque C · Tu logger (copia los valores en la pestaña «Práctica», columna E)
equipo = 1  # @param {type:"slider", min:1, max:10, step:1}
coinciden = [k for k in tabla["Logger"] if re.match(r"(?i)eq0?%d(\D|$)" % equipo, k)]
if not coinciden:
    raise SystemExit(f"No encuentro el archivo del equipo {equipo}: debe llamarse Eq{equipo:02d}_….csv")
clave = coinciden[0]
f = tabla[tabla["Logger"] == clave].iloc[0]
print(f"Logger {clave} · mediana del grupo (escríbela en «Datos del grupo» de la pestaña Práctica): {mediana:.3f} °C\n")
valores = [int(f["n"]), round(f["Media"], 3), round(f["s"], 4), round(f["u"], 4), round(f["Desviación"], 3), f["Veredicto"]]
for v in valores:
    print(v)


## Antes de irte
Anota en la **ficha de campaña** el sesgo de tu logger (la desviación) y regrésalo a su sitio.